In [31]:
from pathlib import Path
import pandas as pd

In [34]:
YEAR = 2023
PROJECT_DIR = Path("../..").resolve()

print("Base Directory:", PROJECT_DIR)


Base Directory: /Users/nicholasbenelli/Workspace/repos/GitHub/-sports/-tennis/Tennis-Data-Pipeline


## Load Data

### Tennis is My Life

In [36]:


df_timl = pd.read_csv(PROJECT_DIR / f"data/raw/timl/atp/{YEAR}.csv")
print(list(df_timl.columns))
df_timl.head()



['tourney_id', 'tourney_name', 'surface', 'draw_size', 'tourney_level', 'indoor', 'tourney_date', 'match_num', 'winner_id', 'winner_seed', 'winner_entry', 'winner_name', 'winner_hand', 'winner_ht', 'winner_ioc', 'winner_age', 'winner_rank', 'winner_rank_points', 'loser_id', 'loser_seed', 'loser_entry', 'loser_name', 'loser_hand', 'loser_ht', 'loser_ioc', 'loser_age', 'loser_rank', 'loser_rank_points', 'score', 'best_of', 'round', 'minutes', 'w_ace', 'w_df', 'w_svpt', 'w_1stIn', 'w_1stWon', 'w_2ndWon', 'w_SvGms', 'w_bpSaved', 'w_bpFaced', 'l_ace', 'l_df', 'l_svpt', 'l_1stIn', 'l_1stWon', 'l_2ndWon', 'l_SvGms', 'l_bpSaved', 'l_bpFaced']


,tourney_id,tourney_name,surface,draw_size,tourney_level,indoor,tourney_date,match_num,winner_id,winner_seed,...,w_bpFaced,l_ace,l_df,l_svpt,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced
0,2023-9900,United Cup,Hard,18,A,O,20230102,1,GH92,NaN,...,0.0,0.0,0.0,81.0,50.0,37.0,15.0,11.0,3.0,4.0
1,2023-9900,United Cup,Hard,18,A,O,20230102,2,CG80,8.0,...,10.0,2.0,1.0,70.0,50.0,31.0,8.0,11.0,5.0,9.0
2,2023-9900,United Cup,Hard,18,A,O,20230102,3,ME82,NaN,...,3.0,2.0,2.0,56.0,37.0,16.0,5.0,6.0,4.0,10.0
3,2023-9900,United Cup,Hard,18,A,O,20230102,4,RC91,6.0,...,0.0,1.0,3.0,57.0,37.0,22.0,10.0,9.0,4.0,7.0
4,2023-9900,United Cup,Hard,18,A,O,20230102,5,GH92,NaN,...,4.0,9.0,1.0,104.0,63.0,51.0,21.0,17.0,3.0,5.0


### Tennis Data UK

In [10]:
df_uk = pd.read_csv(f"../../data/clean/tennis-data-uk/atp/atp_singles_results_{YEAR}.csv")
print(list(df_uk.columns))

df_uk.head()

['ATP', 'Location', 'Tournament', 'Date', 'Series', 'Court', 'Surface', 'Round', 'Best of', 'Winner', 'Loser', 'WRank', 'LRank', 'WPts', 'LPts', 'W1', 'L1', 'W2', 'L2', 'W3', 'L3', 'W4', 'L4', 'W5', 'L5', 'Wsets', 'Lsets', 'Comment', 'B365W', 'B365L', 'PSW', 'PSL', 'MaxW', 'MaxL', 'AvgW', 'AvgL']


,ATP,Location,Tournament,Date,Series,Court,Surface,Round,Best of,Winner,...,Lsets,Comment,B365W,B365L,PSW,PSL,MaxW,MaxL,AvgW,AvgL
0,1,Adelaide,Adelaide International 1,1/1/23,ATP250,Outdoor,Hard,1st Round,3,Giron M.,...,1.0,Completed,1.91,1.91,1.93,1.95,1.99,1.95,1.89,1.89
1,1,Adelaide,Adelaide International 1,1/1/23,ATP250,Outdoor,Hard,1st Round,3,Mcdonald M.,...,0.0,Retired,1.36,3.20,1.39,3.25,1.44,3.40,1.36,3.12
2,1,Adelaide,Adelaide International 1,1/2/23,ATP250,Outdoor,Hard,1st Round,3,Kecmanovic M.,...,0.0,Completed,1.57,2.38,1.58,2.53,1.64,2.53,1.58,2.36
3,1,Adelaide,Adelaide International 1,1/2/23,ATP250,Outdoor,Hard,1st Round,3,Nishioka Y.,...,1.0,Completed,3.75,1.29,4.00,1.28,4.00,1.31,3.56,1.29
4,1,Adelaide,Adelaide International 1,1/2/23,ATP250,Outdoor,Hard,1st Round,3,Popyrin A.,...,0.0,Completed,6.50,1.11,6.20,1.15,6.75,1.18,6.04,1.13


Get Tournaments and their information

In [27]:
df_uk[["ATP", "Location", "Tournament", "Series", "Court", "Surface", "Best of"]].drop_duplicates(keep="first")

,ATP,Location,Tournament,Series,Court,Surface,Best of
0,1,Adelaide,Adelaide International 1,ATP250,Outdoor,Hard,3
31,2,Pune,Maharashtra Open,ATP250,Outdoor,Hard,3
58,3,Adelaide,Adelaide International 2,ATP250,Outdoor,Hard,3
85,4,Auckland,ASB Classic,ATP250,Outdoor,Hard,3
112,5,Melbourne,Australian Open,Grand Slam,Outdoor,Hard,5
...,...,...,...,...,...,...,...
2548,60,Vienna,Vienna Open,ATP500,Indoor,Hard,3
2579,61,Paris,BNP Paribas Masters,Masters 1000,Indoor,Hard,3
2634,62,Metz,Open de Moselle,ATP250,Indoor,Hard,3
2661,63,Sofia,Sofia Open,ATP250,Indoor,Hard,3


In [43]:
# ATP alone isn't a unique tournament id: tennis-data.co.uk reuses the same ATP
# number for tournaments played concurrently in the same week (e.g. ATP 58 is
# both Stockholm's Nordic Open and the Tokyo Japan Open in week 42, 2023).
# Location disambiguates those, so key on (ATP, Location) instead.
tourney_key = ["ATP", "Location"]
info_cols = ["Tournament", "Series", "Court", "Surface", "Best of"]


def mode_or_na(s: pd.Series):
    """Most frequent value in the group (used to smooth over stray data-entry errors)."""
    m = s.mode()
    return m.iloc[0] if not m.empty else pd.NA


# Guard: within each (ATP, Location) key, attributes should be consistent.
# Anything flagged here is a genuine data-quality issue worth reviewing by hand.
nunique_per_key = df_uk.groupby(tourney_key)[info_cols].nunique()
inconsistent = nunique_per_key[(nunique_per_key > 1).any(axis=1)]

if not inconsistent.empty:
    print(f"Warning: {len(inconsistent)} tournament(s) have inconsistent attributes:")
    display(inconsistent)
else:
    print("All tournament attributes are consistent within each (ATP, Location).")

match_dates = pd.to_datetime(df_uk["Date"], format="%m/%d/%y")

tournament_info = df_uk.groupby(tourney_key)[info_cols].agg(mode_or_na)
date_range = match_dates.groupby([df_uk["ATP"], df_uk["Location"]]).agg(
    Start_Date="min", End_Date="max"
)

uk_tournaments = tournament_info.join(date_range).reset_index()
uk_tournaments


,,Tournament,Series,Court,Surface,Best of
ATP,Location,,,,,
38,London,1,1,1,1,2


,ATP,Location,Tournament,Series,Court,Surface,Best of,Start_Date,End_Date
0,1,Adelaide,Adelaide International 1,ATP250,Outdoor,Hard,3,2023-01-01,2023-01-08
1,2,Pune,Maharashtra Open,ATP250,Outdoor,Hard,3,2023-01-02,2023-01-07
2,3,Adelaide,Adelaide International 2,ATP250,Outdoor,Hard,3,2023-01-09,2023-01-14
3,4,Auckland,ASB Classic,ATP250,Outdoor,Hard,3,2023-01-08,2023-01-14
4,5,Melbourne,Australian Open,Grand Slam,Outdoor,Hard,5,2023-01-16,2023-01-29
...,...,...,...,...,...,...,...,...,...
60,60,Vienna,Vienna Open,ATP500,Indoor,Hard,3,2023-10-23,2023-10-29
61,61,Paris,BNP Paribas Masters,Masters 1000,Indoor,Hard,3,2023-10-30,2023-11-05
62,62,Metz,Open de Moselle,ATP250,Indoor,Hard,3,2023-11-05,2023-11-11
63,63,Sofia,Sofia Open,ATP250,Indoor,Hard,3,2023-11-07,2023-11-11


In [45]:
display(df_uk.loc[df_uk["ATP"].isin([38]), ["ATP", "Location", *info_cols, "Date"]].drop_duplicates())


,ATP,Location,Tournament,Series,Court,Surface,Best of,Date
1547,38,London,Wimbledon,Grand Slam,Outdoor,Grass,5,7/3/23
1571,38,London,Wimbledon,Grand Slam,Outdoor,Grass,5,7/4/23
1576,38,London,Wimbledon,Grand Slam,Outdoor,Grass,5,7/5/23
1601,38,London,Wimbledon,Grand Slam,Outdoor,Grass,5,7/6/23
1632,38,London,Wimbledon,Grand Slam,Outdoor,Grass,5,7/7/23
1651,38,London,Wimbledon,Grand Slam,Outdoor,Grass,5,7/8/23
1658,38,London,Wimbledon,Grand Slam,Outdoor,Grass,3,7/9/23
1659,38,London,Wimbledon,Grand Slam,Outdoor,Grass,5,7/9/23
1662,38,London,Wimbledon,Grand Slam,Outdoor,Grass,5,7/10/23
1667,38,London,Wimbledon,Grand Slam,Outdoor,Grass,5,7/11/23


In [13]:
uk_tourneys = df_uk['Tournament'].unique()
print(uk_tourneys)

<StringArray>
[                    'Adelaide International 1',
                             'Maharashtra Open',
                     'Adelaide International 2',
                                  'ASB Classic',
                              'Australian Open',
                                 'Cordoba Open',
                                  'Dallas Open',
                           'Open Sud de France',
                               'Argentina Open',
                            'Delray Beach Open',
             'ABN AMRO World Tennis Tournament',
                       'Qatar Exxon Mobil Open',
                                      'Open 13',
                                     'Rio Open',
                             'Abierto Mexicano',
                   'Dubai Tennis Championships',
                                   'Chile Open',
                             'BNP Paribas Open',
                                   'Miami Open',
                                 'Estoril Open',
      

In [20]:
df_timl.loc[~df_timl['tourney_name'].str.contains("Davis Cup"), "tourney_name"].unique()

<StringArray>
[          'United Cup',           'Adelaide-1',                 'Pune',
             'Auckland',           'Adelaide-2',      'Australian Open',
              'Cordoba',               'Dallas',          'Montpellier',
         'Delray Beach',         'Buenos Aires',            'Rotterdam',
                 'Doha',       'Rio De Janeiro',            'Marseille',
             'Santiago',             'Acapulco',                'Dubai',
 'Indian Wells Masters',        'Miami Masters',              'Estoril',
              'Houston',            'Marrakech',  'Monte Carlo Masters',
            'Barcelona',               'Munich',           'Banja Luka',
       'Madrid Masters',         'Rome Masters',               'Geneva',
                 'Lyon',        'Roland Garros',     ''s-Hertogenbosch',
            'Stuttgart',                'Halle',         'Queen's Club',
             'Mallorca',           'Eastbourne',            'Wimbledon',
               'Gstaad',             

In [21]:
source_df = df_uk.copy()
canonical_df = df_timl.loc[~df_timl['tourney_name'].str.contains("Davis Cup")].copy()


source_tournaments = (
    source_df[["Tournament"]]
    .drop_duplicates()
    .rename(columns={"Tournament": "source_tournament_name"})
)

canonical_tournaments = (
    canonical_df[["tourney_id", "tourney_name"]]
    .drop_duplicates()
    .rename(
        columns={
            "tourney_id": "canonical_tournament_id",
            "tourney_name": "canonical_tournament_name",
        }
    )
)

tournament_crosswalk = source_tournaments.merge(
    canonical_tournaments,
    left_on="source_tournament_name",
    right_on="canonical_tournament_name",
    how="left",
    validate="one_to_one",
)

tournament_crosswalk["match_method"] = "exact_name"
tournament_crosswalk["confidence"] = 1.0
tournament_crosswalk["review_flag"] = (
    tournament_crosswalk["canonical_tournament_id"].isna()
)

In [22]:
tournament_crosswalk

,source_tournament_name,canonical_tournament_id,canonical_tournament_name,match_method,confidence,review_flag
0,Adelaide International 1,NaN,NaN,exact_name,1.0,True
1,Maharashtra Open,NaN,NaN,exact_name,1.0,True
2,Adelaide International 2,NaN,NaN,exact_name,1.0,True
3,ASB Classic,NaN,NaN,exact_name,1.0,True
4,Australian Open,2023-580,Australian Open,exact_name,1.0,False
...,...,...,...,...,...,...
59,Vienna Open,NaN,NaN,exact_name,1.0,True
60,BNP Paribas Masters,NaN,NaN,exact_name,1.0,True
61,Open de Moselle,NaN,NaN,exact_name,1.0,True
62,Sofia Open,NaN,NaN,exact_name,1.0,True


In [23]:
tournament_crosswalk["year"] = 2023
tournament_crosswalk["source"] = "tennis_data_uk"

In [24]:
print(tournament_crosswalk)

      source_tournament_name canonical_tournament_id  \
0   Adelaide International 1                     NaN   
1           Maharashtra Open                     NaN   
2   Adelaide International 2                     NaN   
3                ASB Classic                     NaN   
4            Australian Open                2023-580   
..                       ...                     ...   
59               Vienna Open                     NaN   
60       BNP Paribas Masters                     NaN   
61           Open de Moselle                     NaN   
62                Sofia Open                     NaN   
63               Masters Cup                     NaN   

   canonical_tournament_name match_method  confidence  review_flag  year  \
0                        NaN   exact_name         1.0         True  2023   
1                        NaN   exact_name         1.0         True  2023   
2                        NaN   exact_name         1.0         True  2023   
3                      